# AIC 2026 Frame Extraction Worker 2

Production worker for the next 73 corpus videos. `L21_V001` is included only as a resume/checksum probe and is never added to Worker 2's processing queue. Run after Worker 1 has completed `L21_V001`. Select **GPU T4 x2**, enable Internet, attach `lyduchoang/aic-26-video`, and enable Kaggle Secrets `AIC_RCLONE_CONFIG` plus `AIC_GDRIVE_FOLDER_ID`.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import time

SESSION_STARTED_AT = time.time()
REPO_URL = "https://github.com/AIVIETNAM-AIO-Dewey/AIC-2026.git"
BRANCH = "feat/offline-frame-extraction-transnetv2"
TARGET = Path("/kaggle/working/AIC-2026")

def run_streamed(command, *, cwd=None, env=None):
    print("$", " ".join(map(str, command)), flush=True)
    process = subprocess.Popen(
        command, cwd=cwd, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    assert process.stdout is not None
    lines = []
    for line in process.stdout:
        print(line, end="", flush=True)
        lines.append(line.rstrip())
    if process.wait() != 0:
        raise RuntimeError("Repository command failed:\n" + "\n".join(lines[-80:]))

clone_env = os.environ.copy()
try:
    from kaggle_secrets import UserSecretsClient
    github_token = UserSecretsClient().get_secret("AIC_GITHUB_TOKEN")
except Exception:
    github_token = None
if github_token:
    clone_env.update({
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": f"Authorization: Bearer {github_token}",
    })
print(f"[repo] phase=sync result=attempting branch={BRANCH}", flush=True)
if TARGET.exists() and not (TARGET / ".git").is_dir():
    raise RuntimeError(f"Target exists but is not a git repository: {TARGET}")
if not TARGET.exists():
    run_streamed(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(TARGET)], env=clone_env)
else:
    run_streamed(["git", "fetch", "origin", BRANCH], cwd=TARGET, env=clone_env)
    run_streamed(["git", "switch", BRANCH], cwd=TARGET, env=clone_env)
    run_streamed(["git", "pull", "--ff-only", "origin", BRANCH], cwd=TARGET, env=clone_env)
os.chdir(TARGET)
if str(TARGET) not in sys.path:
    sys.path.insert(0, str(TARGET))
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print(f"[repo] phase=sync result=success branch={BRANCH} commit={commit}", flush=True)


In [ ]:
import json
import sys

WORKER_ID = "frame_extraction_worker_2"
RESUME_PROBE_VIDEO_IDS = ["L21_V001"]
OWNED_VIDEO_IDS = [
    "L23_V014", "L23_V015", "L23_V016", "L23_V017", "L23_V018",
    "L23_V019", "L23_V020", "L23_V021", "L23_V022", "L23_V023",
    "L23_V024", "L23_V025",
    "L24_V002", "L24_V003", "L24_V004", "L24_V005", "L24_V006",
    "L24_V007", "L24_V008", "L24_V009", "L24_V010", "L24_V011",
    "L24_V012", "L24_V013", "L24_V014", "L24_V015", "L24_V016",
    "L24_V017", "L24_V018", "L24_V019", "L24_V020", "L24_V021",
    "L24_V022", "L24_V023", "L24_V024", "L24_V025", "L24_V026",
    "L24_V027", "L24_V028", "L24_V029", "L24_V030", "L24_V031",
    "L24_V032", "L24_V033", "L24_V035", "L24_V036", "L24_V037",
    "L24_V038", "L24_V039", "L24_V040", "L24_V041", "L24_V042",
    "L24_V043", "L24_V044", "L24_V045",
    "L25_V001", "L25_V002", "L25_V003", "L25_V004", "L25_V005",
    "L25_V006", "L25_V007", "L25_V008", "L25_V009", "L25_V010",
    "L25_V011", "L25_V012", "L25_V013", "L25_V014", "L25_V015",
    "L25_V016", "L25_V017", "L25_V018",
]
if len(OWNED_VIDEO_IDS) != 73 or len(set(OWNED_VIDEO_IDS)) != 73:
    raise AssertionError(f"Worker 2 must own 73 unique videos, got {len(OWNED_VIDEO_IDS)}")
if set(OWNED_VIDEO_IDS) & set(RESUME_PROBE_VIDEO_IDS):
    raise AssertionError("Resume probe must not enter the Worker 2 processing queue")
with open("configs/master_video_list.txt", encoding="utf-8") as stream:
    master_ids = [line.strip() for line in stream if line.strip()]
expected_worker_2 = master_ids[73:146]
if OWNED_VIDEO_IDS != expected_worker_2:
    raise AssertionError("Worker 2 assignment does not match master-list positions 74-146")
print(f"[assignment] worker={WORKER_ID} owned={len(OWNED_VIDEO_IDS)}", flush=True)
print(f"[assignment] resume_probe={RESUME_PROBE_VIDEO_IDS}", flush=True)
print(f"[assignment] first={OWNED_VIDEO_IDS[0]} last={OWNED_VIDEO_IDS[-1]}", flush=True)

from scripts.kaggle_frame_worker_bootstrap import run_kaggle_worker
WORKER_REPORT = run_kaggle_worker(
    worker_id=WORKER_ID,
    owned_video_ids=OWNED_VIDEO_IDS,
    resume_probe_video_ids=RESUME_PROBE_VIDEO_IDS,
    session_started_at=SESSION_STARTED_AT,
)


In [ ]:
print(f"[final] worker_id={WORKER_ID}", flush=True)
print(f"[final] status={WORKER_REPORT['status']}", flush=True)
print(f"[final] completed={len(WORKER_REPORT.get('remote_completed', []))}/73", flush=True)
print(f"[final] remaining={WORKER_REPORT.get('remaining', [])}", flush=True)
print(f"[final] resume_probes={WORKER_REPORT.get('resume_probes')}", flush=True)
print(f"[final] drive_url={WORKER_REPORT.get('drive_url')}", flush=True)
print(f"[final] benchmark_remote=self-cut-btc-compatible/benchmark/{WORKER_ID}.json", flush=True)
